# RealSaS — Knight V9 G3B Residual Classification — Run All

**Diagnostic-only companion to the completed V9 static-mechanics notebook. CPU-only.**

This notebook does **not** rerun flip/collapse/cavity repair. It hash-verifies the completed Drive result, reruns only the existing G3B compatibility proof, classifies the 48 residual unsafe faces, and stores the report beside the completed result.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

BOOTSTRAP_COMMIT = 'e3d1ceabb87bfbdc5b17baad51a450760aa073da'
BOOTSTRAP = Path('/content/realsas_v9_g3b_classifier_bootstrap')
if BOOTSTRAP.exists():
    shutil.rmtree(BOOTSTRAP)
subprocess.run(['git','clone','--quiet','https://github.com/merynz/RealSaS-OPT.git',str(BOOTSTRAP)],check=True)
subprocess.run(['git','-C',str(BOOTSTRAP),'checkout','--quiet',BOOTSTRAP_COMMIT],check=True)
got=subprocess.check_output(['git','-C',str(BOOTSTRAP),'rev-parse','HEAD'],text=True).strip()
assert got==BOOTSTRAP_COMMIT,(got,BOOTSTRAP_COMMIT)
RUNNER=BOOTSTRAP/'tools/notebooks/run_knight_v9_g3b_residual_classification_notebook_v1.py'
assert RUNNER.is_file(),RUNNER
subprocess.run([sys.executable,'-m','py_compile',str(RUNNER)],check=True)
print('G3B_CLASSIFIER_BOOTSTRAP_PREFLIGHT_PASS', got)


In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/RealSaS_SUBJECT2_KNIGHT_DEMO_V2_20260924/KNIGHT_V9_STATIC_MECHANICS_NOTEBOOK_20261004'
cmd=[sys.executable,'-u',str(RUNNER),'--drive-base',DRIVE_BASE]
print('G3B_CLASSIFIER_RUN_ALL_COMMAND', ' '.join(cmd), flush=True)
subprocess.run(cmd,cwd=BOOTSTRAP,check=True)


In [ ]:
import json
from pathlib import Path

latest=json.loads((Path(DRIVE_BASE)/'RESULTS/LATEST_RUN.json').read_text(encoding='utf-8'))
out=Path(latest['result_dir'])/'G3B_RESIDUAL_CLASSIFICATION_V1'
report=json.loads((out/'G3B_RESIDUAL_CLASSIFICATION.json').read_text(encoding='utf-8'))
seal=json.loads((out/'CLASSIFICATION_SEAL.json').read_text(encoding='utf-8'))
print('\n=== VERIFIED G3B RESIDUAL RESULT ===')
print('unsafe_face_count:', report['summary']['unsafe_face_count'])
print('all_face_unsafe_count:', report['summary']['all_face_unsafe_count'])
print('dynamic_failure_signature_counts:', report['summary']['dynamic_failure_signature_counts'])
print('rest_static_bad_count:', report['summary']['rest_static_bad_count'])
print('dynamic_only_count:', report['summary']['dynamic_only_count'])
print('seam_face_count:', report['summary']['seam_face_count'])
print('classification_sha256:', seal['classification_sha256'])
try:
    import pandas as pd
    display(pd.DataFrame(report['faces']))
except Exception:
    print(json.dumps(report['faces'],indent=2,sort_keys=True))
print('KNIGHT_V9_G3B_RESIDUAL_CLASSIFICATION_NOTEBOOK_VERIFIED_PASS')
